In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

ruta = "/content/drive/MyDrive/Máster Ciencia de Datos UOC/TFM/desarrollo"
os.chdir(ruta)

In [3]:
!pip install scikeras

In [4]:
!pip install --upgrade scikit-learn scikeras tensorflow

In [5]:
import pandas as pd
import model_utils
import random
import joblib
import seaborn as sns

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
from scikeras.wrappers import KerasRegressor
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

SEED = 7
random.seed(SEED)

# Proporciones del dataset
train_size = 0.8
test_size = 0.2


param_grid = {
    "model__units": [25, 50, 100],
    "model__optimizer": ["adam", "sgd"],
    "epochs": [10, 50],
    "batch_size": [16, 32]
}

cv = TimeSeriesSplit(n_splits=10)

In [10]:
def build_lstm_model(units=50, optimizer="adam"):
    model = Sequential()

    model.add(LSTM(units=units, input_shape=(n_predictors, n_features)))
    model.add(Dense(1))

    model.compile(optimizer=optimizer, loss="mse")

    return model

lstm_regressor = KerasRegressor(
    model=build_lstm_model,
    verbose=0
)

In [11]:
df_7d_no_info = pd.read_csv(r'./qDiario_7d_no_info.csv',parse_dates=["Fecha"], index_col=["Fecha"])

In [12]:
X_train_7d_no_info, X_test_7d_no_info, y_train_7d_no_info, y_test_7d_no_info = model_utils.split_train_test_deep(df_7d_no_info,
                                                train_size_=train_size,test_size_=test_size,
                                                target="m3(t)", _scale=True,verbose=False)

n_predictors = X_train_7d_no_info.shape[1]
n_features = 1

In [13]:
grid = GridSearchCV(
    estimator=lstm_regressor,
    param_grid=param_grid,
    cv=cv,
    n_jobs=-1
)

grid_result = grid.fit(X_train_7d_no_info, y_train_7d_no_info, callbacks=[early_stop], validation_split=0.2, verbose=1)

Epoch 1/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0306 - val_loss: 0.0169
Epoch 2/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0125 - val_loss: 0.0152
Epoch 3/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0120 - val_loss: 0.0146
Epoch 4/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0122 - val_loss: 0.0143
Epoch 5/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0120 - val_loss: 0.0144
Epoch 6/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0116 - val_loss: 0.0140
Epoch 7/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0117 - val_loss: 0.0141
Epoch 8/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0117 - val_loss: 0.0136
Epoch 9/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0113 - val_loss: 0.0135
Epoch 10/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0109 - val_loss: 0.0132
Epoch 11/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0109 - val_loss: 0.0132
Epoch 12/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0109 -